# Replication Analysis using FinnGen GWAS Summary Statistics
This notebook processes FinnGen GWAS summary statistics for heart failure, coronary artery disease, and myocardial infarction. FinnGen is used as an independent replication resource to check whether gene-region variants also appear in comparable cardiovascular phenotypes.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
gene_regions_file = Path("../data/interim/ensembl/gene_regions_grch38.csv")

finngen_hf_path = Path("../data/raw/finngen/finngen_hf.tsv")
finngen_cad_path = Path("../data/raw/finngen/finngen_cad.tsv")
finngen_mi_path = Path("../data/raw/finngen/finngen_mi.tsv")

out_dir = Path("../data/interim/finngen")
out_dir.mkdir(parents=True, exist_ok=True)

out_hf_csv = out_dir / "hf_associations.csv"
out_cad_csv = out_dir / "cad_associations.csv"
out_mi_csv = out_dir / "mi_associations.csv"

In [3]:
gene_regions = pd.read_csv(gene_regions_file, sep=";")

gene_regions.head()

,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22640201,1,+,GRCh38,50000,22585057,22690201,22585057,22635056,22640202,22690201,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648122,1,+,GRCh38,50000,22592558,22698122,22592558,22642557,22648123,22698122,protein_coding,Gene,Ensembl REST


In [4]:
genes_for_finngen = gene_regions.loc[
    gene_regions["lookup_status"] == "found"
].copy()

genes_for_finngen = (
    genes_for_finngen
    .dropna(subset=["official_gene_symbol", "chromosome", "region_start", "region_end"])
    .drop_duplicates(subset=["official_gene_symbol"])
    .reset_index(drop=True)
)

print("Genes available for FinnGen filtering:", len(genes_for_finngen))

genes_for_finngen.head()

Genes available for FinnGen filtering: 52


,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22640201,1,+,GRCh38,50000,22585057,22690201,22585057,22635056,22640202,22690201,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648122,1,+,GRCh38,50000,22592558,22698122,22592558,22642557,22648123,22698122,protein_coding,Gene,Ensembl REST


In [5]:
for path in [finngen_hf_path, finngen_cad_path, finngen_mi_path]:
    if not path.exists():
        raise FileNotFoundError(f"FinnGen file not found: {path}")

In [6]:
def normalize_chromosome(value):
    value = str(value).strip()
    value = value.replace("chr", "")

    if value.endswith(".0"):
        value = value[:-2]

    return value

In [7]:
def annotate_location_relative_to_gene(position, gene_start, gene_end, strand):
    """
    Annotate whether a variant is inside the gene body,
    upstream, or downstream, taking gene strand into account.
    """
    if pd.isna(position) or pd.isna(gene_start) or pd.isna(gene_end):
        return None

    position = int(position)
    gene_start = int(gene_start)
    gene_end = int(gene_end)

    if gene_start <= position <= gene_end:
        return "intragenic"

    if str(strand) in ["1", "+", "+1"]:
        if position < gene_start:
            return "upstream"
        return "downstream"

    if str(strand) in ["-1", "-", "−"]:
        if position > gene_end:
            return "upstream"
        return "downstream"

    return None


def calculate_distance_from_gene(position, gene_start, gene_end):
    """
    Calculate distance from the gene body.
    Variants inside the gene body have distance 0.
    """
    if pd.isna(position) or pd.isna(gene_start) or pd.isna(gene_end):
        return None

    position = int(position)
    gene_start = int(gene_start)
    gene_end = int(gene_end)

    if gene_start <= position <= gene_end:
        return 0

    if position < gene_start:
        return gene_start - position

    return position - gene_end

In [8]:
regions_for_filtering = genes_for_finngen.copy()

regions_for_filtering["chromosome_filter"] = (
    regions_for_filtering["chromosome"]
    .astype(str)
    .str.replace("chr", "", regex=False)
)

regions_for_filtering["region_start"] = pd.to_numeric(
    regions_for_filtering["region_start"],
    errors="coerce"
)

regions_for_filtering["region_end"] = pd.to_numeric(
    regions_for_filtering["region_end"],
    errors="coerce"
)

regions_for_filtering.head()

,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source,chromosome_filter
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST,19
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST,15
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22640201,1,+,GRCh38,50000,22585057,22690201,22585057,22635056,22640202,22690201,protein_coding,Gene,Ensembl REST,1
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST,1
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648122,1,+,GRCh38,50000,22592558,22698122,22592558,22642557,22648123,22698122,protein_coding,Gene,Ensembl REST,1


In [9]:
finngen_columns = [
    "#chrom",
    "pos",
    "ref",
    "alt",
    "rsids",
    "nearest_genes",
    "pval",
    "mlogp",
    "beta",
    "sebeta",
    "af_alt",
    "af_alt_cases",
    "af_alt_controls"
]

In [10]:
def process_finngen_dataset(input_path):
    matched_chunks = []

    metadata_columns = [
        "query_source",
        "query_value",
        "region_assembly",
        "gene_start",
        "gene_end",
        "region_start",
        "region_end",
        "strand",
        "strand_symbol",
        "location_relative_to_gene",
        "distance_from_gene",
    ]

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        usecols=finngen_columns,
        chunksize=500_000,
        low_memory=False
    ):
        chunk = chunk.rename(columns={"#chrom": "chrom"})

        chunk["chrom_filter"] = chunk["chrom"].apply(normalize_chromosome)
        chunk["pos_filter"] = pd.to_numeric(chunk["pos"], errors="coerce")

        for _, gene_region in regions_for_filtering.iterrows():
            region_chromosome = gene_region["chromosome_filter"]
            region_start = int(gene_region["region_start"])
            region_end = int(gene_region["region_end"])

            matched = chunk[
                (chunk["chrom_filter"] == region_chromosome)
                & (chunk["pos_filter"].between(region_start, region_end))
            ].copy()

            if not matched.empty:
                matched["query_source"] = "gene_region"
                matched["query_value"] = gene_region["official_gene_symbol"]
                matched["region_assembly"] = gene_region["assembly_name"]
                matched["gene_start"] = int(gene_region["gene_start"])
                matched["gene_end"] = int(gene_region["gene_end"])
                matched["region_start"] = region_start
                matched["region_end"] = region_end
                matched["strand"] = gene_region["strand"]
                matched["strand_symbol"] = gene_region["strand_symbol"]

                matched["location_relative_to_gene"] = matched["pos_filter"].apply(
                    lambda position: annotate_location_relative_to_gene(
                        position,
                        gene_region["gene_start"],
                        gene_region["gene_end"],
                        gene_region["strand"]
                    )
                )

                matched["distance_from_gene"] = matched["pos_filter"].apply(
                    lambda position: calculate_distance_from_gene(
                        position,
                        gene_region["gene_start"],
                        gene_region["gene_end"]
                    )
                )

                matched = matched.drop(columns=["chrom_filter", "pos_filter"])

                matched_chunks.append(matched)

    if matched_chunks:
        finngen_associations_df = pd.concat(matched_chunks, ignore_index=True)
    else:
        finngen_associations_df = pd.DataFrame(columns=[
            col.replace("#chrom", "chrom") for col in finngen_columns
        ] + metadata_columns)

    rows_before_deduplication = len(finngen_associations_df)

    finngen_associations_df = finngen_associations_df.drop_duplicates()

    rows_after_deduplication = len(finngen_associations_df)
    duplicate_rows_removed = rows_before_deduplication - rows_after_deduplication

    finngen_associations_df = finngen_associations_df.rename(columns={
        "chrom": "FINNGEN_CHR",
        "pos": "FINNGEN_BP",
        "ref": "FINNGEN_ref",
        "alt": "FINNGEN_alt",
        "rsids": "FINNGEN_rsids",
        "nearest_genes": "FINNGEN_nearest_genes",
        "pval": "FINNGEN_p_value",
        "mlogp": "FINNGEN_mlogp",
        "beta": "FINNGEN_beta",
        "sebeta": "FINNGEN_std_error",
        "af_alt": "FINNGEN_alt_allele_freq",
        "af_alt_cases": "FINNGEN_alt_allele_freq_cases",
        "af_alt_controls": "FINNGEN_alt_allele_freq_controls"
    })

    numeric_cols = [
        "FINNGEN_BP",
        "FINNGEN_p_value",
        "FINNGEN_mlogp",
        "FINNGEN_beta",
        "FINNGEN_std_error",
        "FINNGEN_alt_allele_freq",
        "FINNGEN_alt_allele_freq_cases",
        "FINNGEN_alt_allele_freq_controls",
        "gene_start",
        "gene_end",
        "region_start",
        "region_end",
        "distance_from_gene",
    ]

    for col in numeric_cols:
        if col in finngen_associations_df.columns:
            finngen_associations_df[col] = pd.to_numeric(
                finngen_associations_df[col],
                errors="coerce"
            )

    finngen_associations_df = finngen_associations_df.sort_values(
        ["query_value", "FINNGEN_p_value"],
        ascending=[True, True],
        na_position="last"
    ).reset_index(drop=True)

    return finngen_associations_df, duplicate_rows_removed

In [11]:
finngen_hf_df, hf_duplicate_rows_removed = process_finngen_dataset(
    input_path=finngen_hf_path
)

finngen_hf_df.to_csv(out_hf_csv, index=False, sep=";")

finngen_hf_df.shape

KeyboardInterrupt: 

In [12]:
finngen_cad_df, cad_duplicate_rows_removed = process_finngen_dataset(
    input_path=finngen_cad_path
)

finngen_cad_df.to_csv(out_cad_csv, index=False, sep=";")

finngen_cad_df.shape

(53855, 24)

In [13]:
finngen_mi_df, mi_duplicate_rows_removed = process_finngen_dataset(
    input_path=finngen_mi_path
)

finngen_mi_df.to_csv(out_mi_csv, index=False, sep=";")

finngen_mi_df.shape

(53853, 24)

In [14]:
hf_genes_with = set(finngen_hf_df["query_value"].dropna().unique())
cad_genes_with = set(finngen_cad_df["query_value"].dropna().unique())
mi_genes_with = set(finngen_mi_df["query_value"].dropna().unique())
all_genes = set(genes_for_finngen["official_gene_symbol"])

summary = {
    "region_assembly": "GRCh38",
    "genes_available_for_filtering": int(len(genes_for_finngen)),
    "hf": {
        "phenotype": "Heart failure",
        "finngen_endpoint": "I9_HEARTFAIL",
        "genes_with_hf_associations": int(finngen_hf_df["query_value"].nunique()),
        "genes_without_hf_associations": sorted(all_genes - hf_genes_with),
        "associations": int(len(finngen_hf_df)),
        "unique_rsIDs": int(finngen_hf_df["FINNGEN_rsids"].nunique()),
        "duplicate_rows_removed": int(hf_duplicate_rows_removed),
    },
    "cad": {
        "phenotype": "Coronary artery disease",
        "finngen_endpoint": "I9_CHD",
        "genes_with_cad_associations": int(finngen_cad_df["query_value"].nunique()),
        "genes_without_cad_associations": sorted(all_genes - cad_genes_with),
        "associations": int(len(finngen_cad_df)),
        "unique_rsIDs": int(finngen_cad_df["FINNGEN_rsids"].nunique()),
        "duplicate_rows_removed": int(cad_duplicate_rows_removed),
    },
    "mi": {
        "phenotype": "Myocardial infarction",
        "finngen_endpoint": "I9_MI_STRICT",
        "genes_with_mi_associations": int(finngen_mi_df["query_value"].nunique()),
        "genes_without_mi_associations": sorted(all_genes - mi_genes_with),
        "associations": int(len(finngen_mi_df)),
        "unique_rsIDs": int(finngen_mi_df["FINNGEN_rsids"].nunique()),
        "duplicate_rows_removed": int(mi_duplicate_rows_removed),
    },
}

summary

{'region_assembly': 'GRCh38',
 'genes_available_for_filtering': 52,
 'hf': {'phenotype': 'Heart failure',
  'finngen_endpoint': 'I9_HEARTFAIL',
  'genes_with_hf_associations': 51,
  'genes_without_hf_associations': ['TIMP1'],
  'associations': 53855,
  'unique_rsIDs': 49078,
  'duplicate_rows_removed': 0},
 'cad': {'phenotype': 'Coronary artery disease',
  'finngen_endpoint': 'I9_CHD',
  'genes_with_cad_associations': 51,
  'genes_without_cad_associations': ['TIMP1'],
  'associations': 53855,
  'unique_rsIDs': 49078,
  'duplicate_rows_removed': 0},
 'mi': {'phenotype': 'Myocardial infarction',
  'finngen_endpoint': 'I9_MI_STRICT',
  'genes_with_mi_associations': 51,
  'genes_without_mi_associations': ['TIMP1'],
  'associations': 53853,
  'unique_rsIDs': 49077,
  'duplicate_rows_removed': 0}}